# DeiT-Tiny ImageNet + CBAM — Grad-CAM trên Validation V08

Notebook giải thích các checkpoint **đã train** của `EXP-VIT-DEITTINY-CBAM-TRANSFER-5SEEDS-012`; không train lại. Mặc định chạy cả 5 seed của biến thể CBAM, lấy ảnh chỉ từ Validation V08, và lưu heatmap cùng manifest trong thư mục riêng cạnh kết quả thí nghiệm. Test và historical Holdout bị khóa.

DeiT này dùng CLS token, distillation token và 196 patch tokens. Patch tokens được đưa thành lưới 14×14; Grad-CAM tính gradient của **class logit** theo đầu ra 14×14 **sau CBAM**. Vì đầu phân loại nối CLS với pooled patch features, heatmap chỉ giải thích đóng góp của nhánh patch, không bao quát toàn bộ CLS pathway. CBAM spatial gate được vẽ riêng; nó không phải Grad-CAM hay mặt nạ vùng bệnh.

Phương pháp dựa trên [Keras Grad-CAM](https://keras.io/examples/vision/grad_cam/) và cách diễn giải patch-grid cho transformer trong [pytorch-grad-cam ViT guide](https://github.com/jacobgil/pytorch-grad-cam/blob/master/tutorials/vision_transformers.md). Đây là hình giải thích nghiên cứu, **không phải định vị tổn thương được kiểm chứng**.

Chạy các cell từ đầu trong Colab. Notebook dùng tài khoản train có lối tắt `Cashew_Leaf_model_result` trong My Drive, giống notebook EXP-012. Có thể chạy CPU nhưng GPU nhanh hơn.


## 01 — Cấu hình

In [ ]:
EXPERIMENT_ID = 'EXP-VIT-DEITTINY-CBAM-TRANSFER-5SEEDS-012'
CLASS_NAMES = ('anthracnose', 'healthy', 'leaf_miner', 'red_rust', 'not_cashew_leaf')
ALL_SEEDS = (42, 123, 2026, 3407, 7777)
SEEDS_TO_ANALYZE = ALL_SEEDS  # Đổi thành (42,) để chạy nhanh một seed.
VARIANT = 'cbam'  # 'cbam' hoặc 'baseline_no_cbam' để đối chiếu.
N_CORRECT_PER_CLASS = 1  # Tối đa một ảnh đúng có confidence cao cho mỗi lớp.
N_WRONG = 2  # Ảnh sai có confidence cao nhất của seed đầu tiên.
INCLUDE_TRUE_CLASS_FOR_ERRORS = True
MANUAL_RELATIVE_PATHS = ()  # Ví dụ: ('val/anthracnose/anthracnose_1.jpg',)
MAX_PREVIEWS = 5  # Số hình hiện trực tiếp trong notebook; tất cả vẫn được lưu.

DATASET_ZIP_PATH_OVERRIDE = None  # Đường dẫn tuyệt đối tới CashewData_Split (8).zip nếu tự chọn.
EXPERIMENT_ROOT_OVERRIDE = None  # Thư mục EXP-012 chứa cbam/seed_... nếu tự chọn.
OUTPUT_ROOT_OVERRIDE = None  # Thư mục lưu Grad-CAM; mặc định là thư mục riêng cạnh EXP-012.
FORCE_DRIVE_REMOUNT = False  # True nếu Colab đang mount sai tài khoản Google.

EXPECTED_V08_SHA256 = '75C5928B2F6432ACA901BC16831E91F9AA4BDACC1E2FF536753D71DA60C147E7'
assert VARIANT in ('cbam', 'baseline_no_cbam')
assert SEEDS_TO_ANALYZE and len(set(SEEDS_TO_ANALYZE)) == len(SEEDS_TO_ANALYZE)
assert set(SEEDS_TO_ANALYZE).issubset(ALL_SEEDS)
assert N_CORRECT_PER_CLASS >= 0 and N_WRONG >= 0 and MAX_PREVIEWS >= 0
assert MANUAL_RELATIVE_PATHS or N_CORRECT_PER_CLASS + N_WRONG > 0


## 02 — Mount Drive và xác định đầu vào/đầu ra

In [ ]:
import hashlib, json, os, sys, uuid
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules or bool(os.environ.get('COLAB_RELEASE_TAG'))
if IN_COLAB:
    from google.colab import drive
    mydrive = Path('/content/drive/MyDrive')
    if FORCE_DRIVE_REMOUNT or not mydrive.is_dir():
        try:
            drive.mount('/content/drive', force_remount=FORCE_DRIVE_REMOUNT)
        except Exception as exc:
            raise RuntimeError('Drive mount failed. Reconnect Colab and authorize the training account with the MyDrive shortcut.') from exc
    if not mydrive.is_dir():
        raise RuntimeError('Colab did not expose /content/drive/MyDrive')
    resource_roots = [mydrive / 'Cashew_Leaf_model_result']
    resource_roots += sorted(p for p in mydrive.glob('Cashew_Leaf_model_result*') if p not in resource_roots)
    zip_candidates = [p / 'Dataset' / 'CashewData_Split (8).zip' for p in resource_roots]
    zip_candidates.append(mydrive / 'Dataset' / 'CashewData_Split (8).zip')
    experiment_candidates = [p / 'training_results' / 'current_dataset' / EXPERIMENT_ID for p in resource_roots]
    experiment_candidates.append(mydrive / 'cbam_v08_results' / 'current_dataset' / EXPERIMENT_ID)
else:
    mydrive = None
    zip_candidates = []
    experiment_candidates = []

dataset_zip = (Path(DATASET_ZIP_PATH_OVERRIDE).expanduser() if DATASET_ZIP_PATH_OVERRIDE else
               next((p for p in zip_candidates if p.is_file()), None))
experiment_root = (Path(EXPERIMENT_ROOT_OVERRIDE).expanduser() if EXPERIMENT_ROOT_OVERRIDE else
                   next((p for p in experiment_candidates if
                         (p / VARIANT / f'seed_{SEEDS_TO_ANALYZE[0]}' / 'seed_complete.json').is_file()), None))
if dataset_zip is None or not dataset_zip.is_file():
    raise FileNotFoundError(f'V08 ZIP unreadable. Tried: {[str(p) for p in zip_candidates]}. Set DATASET_ZIP_PATH_OVERRIDE.')
if experiment_root is None or not experiment_root.is_dir():
    raise FileNotFoundError(f'EXP-012 output unreadable. Tried: {[str(p) for p in experiment_candidates]}. Set EXPERIMENT_ROOT_OVERRIDE.')

def verify_writable(folder):
    folder.mkdir(parents=True, exist_ok=True)
    probe = folder / f'.gradcam_write_probe_{uuid.uuid4().hex}'
    try:
        probe.write_text('ok', encoding='utf-8')
    finally:
        probe.unlink(missing_ok=True)

if OUTPUT_ROOT_OVERRIDE:
    output_root = Path(OUTPUT_ROOT_OVERRIDE).expanduser()
    verify_writable(output_root)
else:
    preferred = experiment_root.parent / f'{EXPERIMENT_ID}_GRADCAM'
    try:
        verify_writable(preferred)
        output_root = preferred
    except OSError as exc:
        if mydrive is None:
            raise RuntimeError('Set OUTPUT_ROOT_OVERRIDE to a writable folder') from exc
        output_root = mydrive / 'cbam_v08_gradcam' / EXPERIMENT_ID
        verify_writable(output_root)
        print('Shared results are read-only; saving Grad-CAM on training-account MyDrive:', output_root)

print('V08 ZIP:', dataset_zip)
print('Existing experiment:', experiment_root)
print('Grad-CAM output:', output_root)


## 03 — Nạp Keras và các lớp custom đúng với checkpoint

In [ ]:
import csv, io, importlib, importlib.util, platform, subprocess, zipfile
if IN_COLAB and importlib.util.find_spec('keras_hub') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'keras-hub==0.26.0'])
import numpy as np
import tensorflow as tf
import keras
import keras_hub  # Register KerasHub DeiT backbone and preprocessor for .keras deserialization.
import matplotlib.pyplot as plt
from tensorflow.keras import layers

# Exact serialization-compatible custom layers used by EXP-012.
@keras.utils.register_keras_serializable(package='cashew_cbam')
class ChannelAttention(layers.Layer):
    def __init__(self, reduction_ratio=16, **kwargs):
        super().__init__(**kwargs)
        self.reduction_ratio = int(reduction_ratio)
        if self.reduction_ratio < 1:
            raise ValueError('reduction_ratio must be positive')

    def build(self, input_shape):
        channels = input_shape[-1]
        if channels is None:
            raise ValueError('CBAM requires a known channel dimension')
        self.channels = int(channels)
        hidden = max(1, self.channels // self.reduction_ratio)
        self.fc1 = layers.Dense(hidden, activation='relu', name='shared_fc1')
        self.fc2 = layers.Dense(self.channels, name='shared_fc2')
        super().build(input_shape)

    def attention_map(self, inputs):
        avg = tf.reduce_mean(inputs, axis=(1, 2))
        maximum = tf.reduce_max(inputs, axis=(1, 2))
        logits = self.fc2(self.fc1(avg)) + self.fc2(self.fc1(maximum))
        return tf.reshape(tf.sigmoid(logits), (-1, 1, 1, self.channels))

    def call(self, inputs):
        return inputs * self.attention_map(inputs)

    def get_config(self):
        return {**super().get_config(), 'reduction_ratio': self.reduction_ratio}


@keras.utils.register_keras_serializable(package='cashew_cbam')
class SpatialAttention(layers.Layer):
    def __init__(self, kernel_size=7, **kwargs):
        super().__init__(**kwargs)
        self.kernel_size = int(kernel_size)
        if self.kernel_size < 1 or self.kernel_size % 2 != 1:
            raise ValueError('kernel_size must be positive and odd')
        self.conv = layers.Conv2D(1, self.kernel_size, padding='same', activation='sigmoid')

    def build(self, input_shape):
        self.conv.build((*input_shape[:-1], 2))
        super().build(input_shape)

    def attention_map(self, inputs):
        avg = tf.reduce_mean(inputs, axis=-1, keepdims=True)
        maximum = tf.reduce_max(inputs, axis=-1, keepdims=True)
        return self.conv(tf.concat((avg, maximum), axis=-1))

    def call(self, inputs):
        return inputs * self.attention_map(inputs)

    def get_config(self):
        return {**super().get_config(), 'kernel_size': self.kernel_size}


@keras.utils.register_keras_serializable(package='cashew_cbam')
class CBAM(layers.Layer):
    def __init__(self, reduction_ratio=16, kernel_size=7, **kwargs):
        super().__init__(**kwargs)
        self.reduction_ratio = int(reduction_ratio)
        self.kernel_size = int(kernel_size)
        self.channel = ChannelAttention(reduction_ratio)
        self.spatial = SpatialAttention(kernel_size)

    def build(self, input_shape):
        self.channel.build(input_shape)
        self.spatial.build(input_shape)
        super().build(input_shape)

    def call(self, inputs):
        return self.spatial(self.channel(inputs))

    def get_config(self):
        return {**super().get_config(), 'reduction_ratio': self.reduction_ratio,
                'kernel_size': self.kernel_size}


@keras.utils.register_keras_serializable(package='cashew_cbam')
class TokenSlice(layers.Layer):
    def __init__(self, start, stop=None, **kwargs):
        super().__init__(**kwargs)
        self.start, self.stop = start, stop

    def call(self, tokens):
        return tokens[:, self.start:self.stop, :]

    def get_config(self):
        return {**super().get_config(), 'start': self.start, 'stop': self.stop}


CUSTOM_OBJECTS = {name: cls for name, cls in (
    ('ChannelAttention', ChannelAttention), ('SpatialAttention', SpatialAttention),
    ('CBAM', CBAM), ('TokenSlice', TokenSlice))}
CUSTOM_OBJECTS.update({f'cashew_cbam>{name}': cls for name, cls in list(CUSTOM_OBJECTS.items())})
print('Python', platform.python_version(), 'TensorFlow', tf.__version__,
      'Keras', keras.__version__, 'KerasHub', keras_hub.__version__,
      'GPU', tf.config.list_physical_devices('GPU'))


## 04 — Kiểm tra ZIP V08 và chọn ảnh Validation cố định

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1 << 20), b''):
            digest.update(block)
    return digest.hexdigest()

archive_sha256 = sha256_file(dataset_zip).upper()
if archive_sha256 != EXPECTED_V08_SHA256:
    raise ValueError(f'Wrong V08 ZIP: {archive_sha256}')
print('V08 ZIP SHA256 verified:', archive_sha256)

def prediction_index(seed):
    folder = experiment_root / VARIANT / f'seed_{seed}'
    path = folder / 'predictions_val.csv'
    if not path.is_file():
        raise FileNotFoundError(path)
    with path.open(newline='', encoding='utf-8') as stream:
        rows = list(csv.DictReader(stream))
    if len(rows) != 1430:
        raise ValueError(f'Expected 1430 Validation predictions, found {len(rows)}: {path}')
    indexed = {}
    for row in rows:
        relative = row['relative_path']
        parts = relative.split('/')
        if len(parts) != 3 or parts[0] != 'val' or parts[1] not in CLASS_NAMES or                 row['true_label'] != parts[1] or relative in indexed or                 row['variant'] != VARIANT or int(row['seed']) != seed:
            raise ValueError(f'Invalid Validation prediction row: {relative}')
        indexed[relative] = row
    return indexed

reference = prediction_index(SEEDS_TO_ANALYZE[0])
if MANUAL_RELATIVE_PATHS:
    selected_paths = list(dict.fromkeys(MANUAL_RELATIVE_PATHS))
    missing = set(selected_paths) - set(reference)
    if missing:
        raise ValueError(f'Manual paths are not Validation predictions: {sorted(missing)}')
else:
    selected_paths = []
    for class_name in CLASS_NAMES:
        correct = [row for row in reference.values() if row['true_label'] == class_name
                   and row['correct'].lower() == 'true']
        correct.sort(key=lambda row: (-float(row['confidence']), row['relative_path']))
        selected_paths.extend(row['relative_path'] for row in correct[:N_CORRECT_PER_CLASS])
    wrong = [row for row in reference.values() if row['correct'].lower() == 'false'
             and row['relative_path'] not in selected_paths]
    wrong.sort(key=lambda row: (-float(row['confidence']), row['relative_path']))
    selected_paths.extend(row['relative_path'] for row in wrong[:N_WRONG])
if not selected_paths:
    raise ValueError('No Validation images selected')
with zipfile.ZipFile(dataset_zip) as archive:
    for relative in selected_paths:
        member = 'CashewData_Split/' + relative
        archive.getinfo(member)  # Fail before loading any model if a prediction path is missing.
print('Selected Validation images:', len(selected_paths))
for relative in selected_paths:
    row = reference[relative]
    print(f"{relative}: true={row['true_label']}, seed {SEEDS_TO_ANALYZE[0]} pred={row['pred_label']}, confidence={float(row['confidence']):.3f}")


## 05 — Grad-CAM trên lưới patch sau CBAM

In [ ]:
def load_validation_image(archive, relative_path):
    parts = relative_path.split('/')
    if len(parts) != 3 or parts[0] != 'val' or parts[1] not in CLASS_NAMES or             not parts[2] or parts[2] in ('.', '..'):
        raise ValueError(f'Unsafe or non-Validation path: {relative_path}')
    raw = archive.read('CashewData_Split/' + relative_path)
    image = tf.io.decode_image(raw, channels=3, expand_animations=False)
    image.set_shape((None, None, 3))
    return tf.cast(tf.image.resize(image, (224, 224), method='bilinear'), tf.float32)


def make_cam_model(model):
    name = 'cbam' if VARIANT == 'cbam' else 'patch_grid_14x14'
    layer = model.get_layer(name)
    if tuple(layer.output.shape[1:]) != (14, 14, 192):
        raise ValueError(f'Unexpected {name} output shape: {layer.output.shape}')
    if tuple(model.output.shape[1:]) != (5,):
        raise ValueError(f'Expected five class logits, got {model.output.shape}')
    cam_model = keras.Model(model.inputs, [layer.output, model.output])
    pre_cbam_model = keras.Model(model.inputs, layer.input) if VARIANT == 'cbam' else None
    return cam_model, pre_cbam_model


def gradcam_for_class(cam_model, image, class_index):
    if class_index not in range(len(CLASS_NAMES)):
        raise ValueError(class_index)
    batch = tf.expand_dims(image, 0)
    with tf.GradientTape() as tape:
        feature_map, logits = cam_model(batch, training=False)
        score = logits[:, class_index]
    gradients = tape.gradient(score, feature_map)
    if gradients is None:
        raise RuntimeError('No gradient from class logit to patch-grid feature map')
    if not bool(tf.reduce_all(tf.math.is_finite(gradients))):
        raise RuntimeError('Non-finite Grad-CAM gradients')
    weights = tf.reduce_mean(gradients, axis=(1, 2), keepdims=True)
    raw_cam = tf.nn.relu(tf.reduce_sum(weights * feature_map, axis=-1))[0]
    peak = float(tf.reduce_max(raw_cam).numpy())
    status = 'OK' if peak > 1e-8 else 'ZERO_POSITIVE_CAM'
    cam14 = raw_cam / peak if status == 'OK' else tf.zeros_like(raw_cam)
    cam224 = tf.image.resize(cam14[None, ..., None], (224, 224), method='bilinear')[0, ..., 0]
    cam224 = tf.clip_by_value(cam224, 0.0, 1.0)
    return (cam14.numpy(), cam224.numpy(), status,
            float(tf.reduce_sum(tf.abs(gradients)).numpy()), float(score[0].numpy()))


def cbam_spatial_gate(model, pre_cbam_model, image):
    if pre_cbam_model is None:
        return None
    features = pre_cbam_model(tf.expand_dims(image, 0), training=False)
    cbam = model.get_layer('cbam')
    gate = cbam.spatial.attention_map(cbam.channel(features))
    result = gate.numpy()[0, :, :, 0]
    if result.shape != (14, 14) or not np.isfinite(result).all():
        raise RuntimeError('Invalid CBAM spatial gate')
    return result


def save_cam_figure(image, cam14, cam224, gate14, title, path, show=False):
    rgb = np.clip(image.numpy() / 255.0, 0.0, 1.0)
    heat = plt.colormaps['magma'](cam224)[..., :3]
    overlay = 0.60 * rgb + 0.40 * heat if np.max(cam224) > 0 else rgb
    panels = [('RGB input 224×224', rgb, None), ('Grad-CAM 14×14', cam14, 'magma'),
              ('Grad-CAM overlay', overlay, None)]
    if gate14 is not None:
        panels.append(('CBAM spatial gate (separate)', gate14, 'viridis'))
    fig, axes = plt.subplots(1, len(panels), figsize=(4 * len(panels), 4))
    for ax, (label, value, cmap) in zip(axes, panels):
        ax.imshow(value, cmap=cmap, vmin=0 if cmap else None, vmax=1 if cmap else None)
        ax.set_title(label, fontsize=10)
        ax.axis('off')
    fig.suptitle(title, fontsize=10)
    fig.tight_layout()
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=160, bbox_inches='tight')
    if show:
        plt.show()
    plt.close(fig)


## 06 — Nạp checkpoint, đối chiếu dự đoán và tạo heatmap

In [ ]:
from datetime import datetime, timezone

figures_dir = output_root / VARIANT / 'figures'
maps_dir = output_root / VARIANT / 'maps'
figures_dir.mkdir(parents=True, exist_ok=True)
maps_dir.mkdir(parents=True, exist_ok=True)
records = []
preview_count = 0
with zipfile.ZipFile(dataset_zip) as archive:
    for seed in SEEDS_TO_ANALYZE:
        seed_dir = experiment_root / VARIANT / f'seed_{seed}'
        complete_path = seed_dir / 'seed_complete.json'
        metrics_path = seed_dir / 'validation_metrics.json'
        model_path = seed_dir / 'selected_model.keras'
        if not all(p.is_file() for p in (complete_path, metrics_path, model_path)):
            raise FileNotFoundError(f'Missing completed seed artifacts: {seed_dir}')
        complete = json.loads(complete_path.read_text(encoding='utf-8'))
        metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
        if complete['seed'] != seed or complete['variant'] != VARIANT or                 metrics['seed'] != seed or metrics['variant'] != VARIANT or metrics['split'] != 'val':
            raise ValueError(f'Wrong seed/variant/split provenance: {seed_dir}')
        selected_stage = int(metrics['selected_stage'])
        checkpoint = seed_dir / 'checkpoints' / f'stage{selected_stage}_best.weights.h5'
        if not checkpoint.is_file() or sha256_file(checkpoint) != metrics['checkpoint_sha256'] or                 complete['checkpoint_sha256'] != metrics['checkpoint_sha256']:
            raise RuntimeError(f'Selected checkpoint integrity mismatch: {seed_dir}')
        selected_model_sha = sha256_file(model_path)
        predictions = prediction_index(seed)
        print(f'Seed {seed}: loading selected Stage {selected_stage} model')
        model = keras.models.load_model(model_path, custom_objects=CUSTOM_OBJECTS,
                                       compile=False, safe_mode=True)
        cam_model, pre_cbam_model = make_cam_model(model)
        for relative in selected_paths:
            image = load_validation_image(archive, relative)
            row = predictions[relative]
            true_index = CLASS_NAMES.index(row['true_label'])
            logits = model(tf.expand_dims(image, 0), training=False)
            probability = tf.nn.softmax(logits, axis=-1).numpy()[0]
            predicted_index = int(np.argmax(probability))
            expected = np.array([float(row[f'prob_{name}']) for name in CLASS_NAMES])
            if CLASS_NAMES[predicted_index] != row['pred_label'] or                     not np.allclose(probability, expected, atol=0.005, rtol=0.005):
                raise RuntimeError(f'Saved model disagrees with Validation predictions: seed={seed}, {relative}, '
                                   f'max probability delta={np.max(np.abs(probability - expected)):.5f}')
            gate14 = cbam_spatial_gate(model, pre_cbam_model, image)
            targets = [('predicted', predicted_index)]
            if INCLUDE_TRUE_CLASS_FOR_ERRORS and true_index != predicted_index:
                targets.append(('true', true_index))
            for target_kind, target_index in targets:
                cam14, cam224, status, gradient_l1, class_logit = gradcam_for_class(
                    cam_model, image, target_index)
                image_id = hashlib.sha256(relative.encode('utf-8')).hexdigest()[:12]
                stem = f'seed_{seed}_{image_id}_{target_kind}_{CLASS_NAMES[target_index]}'
                figure_path = figures_dir / f'{stem}.png'
                map_path = maps_dir / f'{stem}.npz'
                np.savez_compressed(map_path, gradcam_14x14=cam14, gradcam_224x224=cam224,
                                    cbam_spatial_gate_14x14=gate14 if gate14 is not None else np.empty((0,)))
                title = (f"{relative} | seed {seed} | true={row['true_label']} | "
                         f"pred={row['pred_label']} ({probability[predicted_index]:.3f}) | "
                         f"CAM target={CLASS_NAMES[target_index]}")
                save_cam_figure(image, cam14, cam224, gate14, title, figure_path,
                                show=preview_count < MAX_PREVIEWS)
                preview_count += 1
                records.append(dict(seed=seed, variant=VARIANT, relative_path=relative,
                    true_label=row['true_label'], pred_label=row['pred_label'],
                    pred_probability=float(probability[predicted_index]),
                    target_kind=target_kind, target_class=CLASS_NAMES[target_index],
                    target_logit=class_logit, target_probability=float(probability[target_index]),
                    cam_status=status, gradient_l1=gradient_l1,
                    selected_stage=selected_stage, checkpoint_sha256=metrics['checkpoint_sha256'],
                    selected_model_sha256=selected_model_sha,
                    figure_path=str(figure_path.relative_to(output_root)),
                    map_path=str(map_path.relative_to(output_root))))
        del cam_model, pre_cbam_model, model
        keras.backend.clear_session()

if not records:
    raise RuntimeError('No Grad-CAM maps were produced')
manifest = output_root / VARIANT / 'gradcam_manifest.csv'
with manifest.open('w', newline='', encoding='utf-8') as stream:
    writer = csv.DictWriter(stream, fieldnames=list(records[0]))
    writer.writeheader()
    writer.writerows(records)
method = ('Grad-CAM on post-CBAM patch grid' if VARIANT == 'cbam' else
          'Grad-CAM on baseline patch grid before GAP')
metadata = dict(source_experiment=EXPERIMENT_ID, method=method,
    variant=VARIANT, seeds=list(SEEDS_TO_ANALYZE), selected_validation_images=selected_paths,
    v08_zip_sha256=archive_sha256, target_policy='predicted; true additionally for errors',
    generated_utc=datetime.now(timezone.utc).isoformat(),
    limitation='Patch-branch class evidence only; no lesion-localization ground truth')
(output_root / VARIANT / 'gradcam_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print('Created', len(records), 'Grad-CAM maps for', len(selected_paths), 'Validation images')
print('Manifest:', manifest)
print('Figures:', figures_dir)
print('Raw maps:', maps_dir)
print('Zero-positive-CAM maps:', sum(r['cam_status'] != 'OK' for r in records))


## 07 — Cách đọc kết quả

- `figures/`: RGB, Grad-CAM 14×14, overlay và spatial gate CBAM (khi có).
- `maps/`: mảng số để tính lại hình mà không phải chạy inference.
- `gradcam_manifest.csv`: ảnh, seed, nhãn thật/dự đoán, lớp mục tiêu, checkpoint hash, trạng thái gradient.
- Heatmap cho lớp dự đoán giải thích **logit dự đoán**; với ảnh sai có thêm heatmap cho lớp thật. Spatial gate là cơ chế attention của CBAM, không thay thế Grad-CAM.
- Heatmap là bằng chứng cục bộ của nhánh patch tại checkpoint đã chọn. Không suy ra rằng vùng sáng chính là tổn thương, và không dùng hình này làm tiêu chí chọn mô hình hay tuyên bố hiệu quả trên Test/Holdout.

Nếu Keras không đọc được `.keras` do lệch phiên bản, hãy so phiên bản TensorFlow/Keras/KerasHub được in ở Cell 03 với `environment.json` trong EXP-012; phiên train đã ghi `tensorflow 2.21.0`, `keras 3.13.2`, `keras_hub 0.26.0`. Không cần train lại để tạo Grad-CAM.
